<a href="https://colab.research.google.com/github/PujaSridhar/finsight/blob/main/notebooks/02_train_qlora.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# FinSight · Phase 2: QLoRA fine-tuning of Mistral-7B-Instruct

**Runtime:** GPU (T4 is enough). Runtime → Change runtime type → T4 GPU.
The default config trains on 4,000 examples for one epoch: roughly 2–3 hours on a T4. Kaggle notebooks
(free T4, 12-hour sessions) are a good alternative if Colab disconnects.

Checkpoints go to Google Drive every 50 steps. If the session drops, re-run the setup cells and the
training cell with `--resume`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Clone the repo and install dependencies (~2 min)
REPO_BRANCH = "main"
!git clone -q -b $REPO_BRANCH https://github.com/PujaSridhar/finsight /content/finsight 2>/dev/null || (cd /content/finsight && git pull -q)
%cd /content/finsight
!pip install -q -r requirements.txt

In [ ]:
# Keep checkpoints and results on Google Drive so a disconnect doesn't lose them.
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/finsight/outputs /content/drive/MyDrive/finsight/results
!rm -rf outputs results && ln -s /content/drive/MyDrive/finsight/outputs outputs && ln -s /content/drive/MyDrive/finsight/results results

In [ ]:
# Mistral-7B-Instruct-v0.3 is gated: accept its terms at
# https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.3, then paste a read token here
# (or store it as the Colab secret HF_TOKEN).
from huggingface_hub import login
login()

In [ ]:
!python -m finsight.data --out data/processed

## Optional: 10-minute smoke test
Catches setup problems (token, GPU memory, library versions) before the long run.

In [ ]:
# !python -m finsight.train --config configs/smoke_test.yaml

## Train

In [ ]:
!python -m finsight.train --config configs/qlora_t4.yaml
# after a disconnect:
# !python -m finsight.train --config configs/qlora_t4.yaml --resume

In [ ]:
import json
from IPython.display import Image, display
s = json.load(open("outputs/finsight-mistral-7b-qlora/training_summary.json"))
print({k: v for k, v in s.items() if k != "log_history"})
display(Image("results/loss_curve.png"))

The adapter is in `outputs/finsight-mistral-7b-qlora/final` (about 160 MB). Next: `03_evaluate.ipynb`.